In [ ]:
#Importo Pacchetti Necessari
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import  MinMaxScaler, OneHotEncoder
from sklearn.metrics import log_loss,accuracy_score, roc_auc_score, silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV 
import requests


In [ ]:

# ─────────────────────────────────────────────
# 1. CONFIGURAZIONE
# ─────────────────────────────────────────────

# Indirizzo locale di Ollama (non cambiare)
OLLAMA_URL = "http://localhost:11434/api/chat"

# Modello da usare — deve essere già scaricato con "ollama pull <nome>"
MODELLO = "phi3:mini"   # cambia con "llama3" o "phi3" se preferisci

# Parola magica per uscire dal loop di feedback
PAROLA_MAGICA = "yes"

# File di output JSON con la memoria dell'agente
OUTPUT_JSON = "context_memory.json"

# ─────────────────────────────────────────────
# 2. LETTURA DEI CSV
# ─────────────────────────────────────────────

def leggi_csv(filepath: str) -> str:
    """
    Legge un file CSV e lo converte in testo tabellare
    da inserire nel prompt come contesto.
    Restituisce stringa vuota se il file non esiste.
    """
    if not os.path.exists(filepath):
        print(f"  [WARN] File non trovato: {filepath} — saltato.")
        return ""

    df = pd.read_csv(filepath, sep=";")
    return df.to_string(index=False)


# ─────────────────────────────────────────────
# 3. CHIAMATA A OLLAMA (via HTTP locale)
# ─────────────────────────────────────────────

def chiedi_a_ollama(cronologia: list) -> str:
    """
    Invia la cronologia dei messaggi a Ollama in locale
    e restituisce la risposta testuale del modello.

    Ollama accetta lo stesso formato di OpenAI:
    [{"role": "user"/"assistant"/"system", "content": "..."}]
    """
    payload = {
        "model": MODELLO,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": 0.6,   # balanced: creative but precise
            "num_predict": 1200   # max response length in tokens
        }
    }

    try:
        risposta = requests.post(OLLAMA_URL, json=payload, timeout=5000)
        risposta.raise_for_status()
        dati = risposta.json()
        return dati["message"]["content"]

    except requests.exceptions.ConnectionError:
        raise ConnectionError(
            "\n❌ Ollama non è in esecuzione!\n"
            "   Avvialo con il comando: ollama serve\n"
            "   Oppure apri l'app Ollama dal menu di sistema."
        )
    except requests.exceptions.Timeout:
        raise TimeoutError(
            "\n❌ Timeout: il modello ha impiegato troppo.\n"
            "   Prova un modello più leggero come 'phi3'."
        )


# ─────────────────────────────────────────────
# 4. COSTRUZIONE DEL PROMPT INIZIALE
# ─────────────────────────────────────────────

def costruisci_prompt_iniziale(general_ctx, business_ctx, features_ctx) -> str:
    """
    Assembla il prompt con tutti e tre i livelli di contesto
    estratti dai CSV. Il modello analizzerà tutto insieme.
    """
    prompt = f"""
You are an expert agent specialized in human context analysis and data interpretation.
You are provided with three levels of contextual information extracted from CSV files.
Your task is to deeply understand these inputs and produce a structured, comprehensive summary that highlights the key insights, 
relationships, and implications for the business environment.

---
## CONTESTO GENERALE
{general_ctx if general_ctx else "Non fornito."}

---
## CONTESTO DI BUSINESS
{business_ctx if business_ctx else "Non fornito."}

---
## CONTESTO DELLE FEATURES E CONCETTI LATENTI
{features_ctx if features_ctx else "Non fornito."}

---
Based on all provided information, deliver the following:

Agent Role
Define the role the agent should assume to provide maximum value.

Focus on analytical, interpretive, and decision‑support capabilities.

General Context

Summarize the overall domain and environment.

Highlight the macro‑level dynamics, actors, and constraints.

Business Context

Describe key objectives, processes, KPIs, and business logic relevant to the scenario.

Identify strategic priorities and operational drivers.

Purpose

Specify what the agent/system must concretely accomplish.

Clarify expected outputs, decisions, or transformations.

Included Features

Explain what each variable measures and what latent concepts they represent.

Highlight how these features relate to the business problem.

Be precise, concise, and structured. Use bullet points where appropriate.
"""
    return prompt.strip()


In [ ]:

# ─────────────────────────────────────────────
# 5. ESTRAZIONE STRUTTURA JSON FINALE
# ─────────────────────────────────────────────

def estrai_struttura_json(risposta_finale: str) -> dict:
    """
    Chiede al modello di estrarre dalla risposta finale
    una struttura JSON pulita da salvare come memoria.
    """
    prompt_json = f"""
From the following analysis, extract a JSON object with the exact structure shown below.
Respond ONLY with valid JSON. And all insights and description should be prompt-designed for the next LLM agent.
No markdown, no comments, no additional text.

{{
  "role": "stringa",
  "general_context": "stringa",
  "business_context": "stringa",
  "scope": "stringa",
  "features_and_concepts": ["stringa", "stringa"]
  "other": "stringa",
}}

Analisi da cui estrarre:
{risposta_finale}
"""

    messaggi = [{"role": "user", "content": prompt_json}]
    testo = chiedi_a_ollama(messaggi)

    # Pulizia: rimuove eventuali backtick markdown che il modello aggiunge
    testo_pulito = testo.strip().strip("```json").strip("```").strip()

    try:
        struttura = json.loads(testo_pulito)
    except json.JSONDecodeError:
        # Fallback sicuro: salva la risposta grezza senza crashare
        print("  [WARN] Il modello non ha restituito JSON valido. Salvo risposta grezza.")
        struttura = {
            "role": "Non estratto automaticamente",
            "general_context": "Non estratto automaticamente",
            "business_context": "Non estratto automaticamente",
            "scope": "Non estratto automaticamente",
            "features_and_concepts": [],
            "other": "Non estratto automaticamente",
            "risposta_completa": risposta_finale
        }

    return struttura


# ─────────────────────────────────────────────
# 6. SALVATAGGIO MEMORIA IN JSON
# ─────────────────────────────────────────────

def salva_memoria(struttura: dict, filepath: str = OUTPUT_JSON):
    """
    Scrive la struttura appresa su disco in formato JSON leggibile.
    """
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(struttura, f, ensure_ascii=False, indent=2)
    print(f"\n✅ Memoria salvata in: {filepath}")


# ─────────────────────────────────────────────
# 7. LOOP PRINCIPALE DELL'AGENTE
# ─────────────────────────────────────────────

def execute_agent_cotext(
    general_csv: str = "general_context.csv",
    business_csv: str = "business_context.csv",
    features_csv: str = "features_context.csv"
):
    """
    Funzione principale:
      1. Legge i tre CSV
      2. Analizza il contesto con Ollama
      3. Mostra la comprensione all'utente
      4. Raccoglie feedback in loop finché non arriva "yes"
      5. Estrae e salva la memoria strutturata in JSON
    """

    print("=" * 60)
    print("  AGENTE CONTESTUALE — powered by Ollama (locale)")
    print(f"  Modello: {MODELLO}")
    print("=" * 60)

    # — STEP 1: Leggi i CSV —
    print("\n📂 Lettura dei file CSV...")
    general_ctx  = leggi_csv(general_csv)
    business_ctx = leggi_csv(business_csv)
    features_ctx = leggi_csv(features_csv)

    # — STEP 2: Primo prompt —
    print("🧠 Invio contesto al modello (potrebbe richiedere qualche secondo)...\n")
    prompt_iniziale = costruisci_prompt_iniziale(
        general_ctx, business_ctx, features_ctx
    )

    # La cronologia mantiene tutta la conversazione in memoria
    cronologia = [{"role": "user", "content": prompt_iniziale}]

    # — STEP 3: Prima risposta —
    risposta = chiedi_a_ollama(cronologia)
    cronologia.append({"role": "assistant", "content": risposta})

    # — STEP 4: Loop di feedback interattivo —
    while True:
        print("\n" + "─" * 60)
        print("📋 COMPRENSIONE CORRENTE DELL'AGENTE:\n")
        print(risposta)
        print("\n" + "─" * 60)
        print(f"\n✏️  Inserisci feedback per affinare (o '{PAROLA_MAGICA}' per confermare):\n")

        feedback = input("Tu → ").strip()

        # Parola magica: esce dal loop
        if feedback.lower() == PAROLA_MAGICA:
            print("\n🎯 Comprensione confermata! Salvataggio in corso...")
            break

        if not feedback:
            print("  [INFO] Nessun testo inserito, riprova.")
            continue

        # Aggiunge il feedback alla cronologia e ottiene nuova risposta
        cronologia.append({"role": "user", "content": feedback})
        print("\n⏳ Aggiornamento comprensione...")
        risposta = chiedi_a_ollama(cronologia)
        cronologia.append({"role": "assistant", "content": risposta})

    # — STEP 5: Estrai JSON e salva —
    print("\n💾 Estrazione struttura finale...")
    struttura = estrai_struttura_json(risposta)
    salva_memoria(struttura)

    # Mostra il risultato finale
    print("\n📦 MEMORIA FINALE:")
    print(json.dumps(struttura, ensure_ascii=False, indent=2))

    return struttura
    



In [ ]:

if __name__ == "__main__":
    execute_agent_cotext(
        general_csv="IF_context_infos/general_context.csv",
        business_csv="IF_context_infos/biz_context.csv",
        features_csv="IF_context_infos/features_context.csv"
    )